# 06 — Baseline: naive forecasts and evaluation

Naive forecasts of the national price and the metrics every model will be judged on.

**Split** (defined once in `src/forecasting.py`, thermal years Oct → Sep):
- **train**: Jan 2021 → Sep 2023 (contains the gas crisis)
- **validation**: Oct 2023 → Sep 2024 (model-structure choices are made here)
- **test**: Oct 2024 → Sep 2025 (looked at once, at the end)

- **Input:** `data/processed/dataset_hourly_2021_2025q3.parquet`
- **Output:** `data/processed/naive_forecasts_2021_2025q3.parquet`, `results/06_baseline_metrics.csv`
- **Key points:** `naive_epf` (yesterday's price for Tue–Fri, last week's for Mon/Sat/Sun) is the benchmark for rMAE. MAPE is excluded (zero-price hours) and so is R².

In [ ]:
# --- Make src/ importable and reload it automatically when it changes ---------
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import forecasting as fc

TARGET = fc.TARGET

# --- Load the dataset and add split / thermal-year columns ---------------------
# The split boundaries live in src/forecasting.py: one definition for all notebooks
dataset = pd.read_parquet(fc.PROCESSED / "dataset_hourly_2021_2025q3.parquet")
dataset = fc.add_split_columns(dataset)

# --- Overview ------------------------------------------------------------------
overview = dataset.groupby("period").agg(
    first_hour=("ts_local", "min"),
    last_hour=("ts_local", "max"),
    n_hours=(TARGET, "size"),
    split=("split", lambda s: "/".join(sorted(s.unique()))),
    mean_price=(TARGET, "mean"),
    std_price=(TARGET, "std"),
)
print(overview.round({"mean_price": 1, "std_price": 1}).to_string())
print()
print(dataset["split"].value_counts().reindex(["train", "validation", "test"]).to_string())

In [ ]:
# --- Daily price profile: one row per local day, one column per local hour ----
# Prices follow PEOPLE (work, heating, lighting), i.e. the local clock: "same
# hour yesterday" is defined on LOCAL hours. The opposite of solar in notebook
# 05, which follows the sun and was shifted in UTC.
dataset["local_date"] = dataset["ts_local"].dt.tz_localize(None).dt.normalize()
dataset["local_hour"] = dataset["ts_local"].dt.hour

# 25-hour days: the two 02:00 rows are averaged into one value.
# 23-hour days: hour 02 does not exist and comes out as NaN.
profile = dataset.pivot_table(
    index="local_date", columns="local_hour", values=TARGET, aggfunc="mean"
)

# Fill the missing 02:00 of spring-DST days halfway between 01:00 and 03:00.
# This value is only ever used as a SOURCE for the next day's naive forecast.
profile = profile.T.interpolate().T

assert profile.shape == (1734, 24), profile.shape
assert profile.notna().all().all()
# shift(k) means "k days earlier" only if no calendar day is missing
day_steps = profile.index.to_series().diff().dropna()
assert (day_steps == pd.Timedelta(days=1)).all()

# --- The four naive forecasts, built on the profile ---------------------------
naive_profiles = {
    "naive_d1": profile.shift(1),                                  # same hour yesterday
    "naive_d7": profile.shift(7),                                  # same hour last week
    "naive_mean7": sum(profile.shift(k) for k in range(1, 8)) / 7,  # mean of the last 7 days
}

# EPF naive (Weron): on Monday, Saturday and Sunday the previous day is a
# different kind of day, so use last week; Tuesday-Friday use yesterday
use_d7 = profile.index.dayofweek.isin([0, 5, 6])   # Monday = 0
naive_epf = naive_profiles["naive_d1"].copy()
naive_epf.loc[use_d7] = naive_profiles["naive_d7"].loc[use_d7]
naive_profiles["naive_epf"] = naive_epf

NAIVE_COLUMNS = list(naive_profiles)

# --- Back to hourly rows ------------------------------------------------------
# Each (local_date, local_hour) row takes its value from the profile. On a
# 25-hour day both 02:00 rows get the same forecast, which is what a trader
# bidding by clock hour would do.
for name, prof in naive_profiles.items():
    long = prof.stack().rename(name).to_frame()   # index: (local_date, local_hour)
    dataset = dataset.merge(long, left_on=["local_date", "local_hour"],
                            right_index=True, how="left")

# --- Checks -------------------------------------------------------------------
assert len(dataset) == 41615
# Naive forecasts must exist on every hour we will ever score
assert dataset.loc[dataset["split"] != "train", NAIVE_COLUMNS].notna().all().all()

# On ordinary days (24 hours today and yesterday) naive_d1 must be exactly the
# price 24 rows earlier. DST days are excluded: there the row arithmetic breaks.
hours_per_day = dataset.groupby("local_date")[TARGET].transform("size")
ordinary = (
    (hours_per_day == 24)
    & (hours_per_day.shift(24) == 24)
    & (dataset["local_date"] - dataset["local_date"].shift(24) == pd.Timedelta(days=1))
)
assert np.allclose(dataset.loc[ordinary, "naive_d1"],
                   dataset[TARGET].shift(24)[ordinary])
print("ordinary hours checked:", int(ordinary.sum()))

# Look at the DST switches: day after a 25-hour day, and day after a 23-hour day
dst_view = dataset["local_date"].isin(pd.to_datetime(["2024-10-28", "2025-03-31"])) \
           & dataset["local_hour"].between(1, 3)
print(dataset.loc[dst_view, ["ts_local", TARGET, "naive_d1"]].to_string(index=False))

In [ ]:
# --- Metrics on validation and test (evaluate() lives in src/forecasting.py) ---
results = pd.concat(
    [fc.evaluate(dataset, NAIVE_COLUMNS, split=split) for split in ["validation", "test"]],
    ignore_index=True,
)

# --- Regression check: moving evaluate() into src/ must not change any number ---
# Compared against the metrics saved by the previous version of this notebook
# (per thermal year, which does not depend on how the split is named)
previous_path = fc.RESULTS / "06_baseline_metrics.csv"
if previous_path.exists():
    previous = pd.read_csv(previous_path)

    def per_year_mae(table: pd.DataFrame) -> pd.Series:
        by_year = table[table["period"] != "all"]
        return by_year.set_index(["period", "forecast"])["MAE"].sort_index()

    old, new = per_year_mae(previous), per_year_mae(results)
    assert old.index.equals(new.index) and np.allclose(old, new), "metrics changed!"
    print("regression check passed: per-year MAE identical to the saved metrics")

# One table per metric: rows = forecasts, columns = splits (one thermal year each)
per_split = results[results["period"] != "all"]
for metric in ["MAE", "RMSE", "WAPE_pct", "rMAE"]:
    print(f"\n{metric}")
    print(per_split.pivot(index="forecast", columns="split", values=metric)
                   [["validation", "test"]].round(2).to_string())

In [ ]:
# --- Where do the naive forecasts fail? MAE by day of week (VALIDATION) -------
# Validation, not test: this analysis informs feature choices for the models,
# and choices must never be based on the test set
validation = dataset.loc[dataset["split"] == "validation"]
day_of_week = validation["ts_local"].dt.day_name()

mae_by_day = pd.DataFrame({
    col: (validation[TARGET] - validation[col]).abs().groupby(day_of_week).mean()
    for col in NAIVE_COLUMNS
})
# groupby sorts alphabetically: put the days back in calendar order
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(mae_by_day.reindex(DAY_ORDER).round(1).to_string())

In [ ]:
# --- Persist naive forecasts and metrics --------------------------------------
# All hours, not only the test: lagged prices like naive_d1 may become model
# features later, and the next notebook must start from files, not from this kernel
fc.RESULTS.mkdir(exist_ok=True)

NAIVE_KEEP = ["ts_utc", "ts_local", "local_date", "local_hour", "period", "split",
              TARGET] + NAIVE_COLUMNS
naive_out = dataset[NAIVE_KEEP].reset_index(drop=True)

naive_path = fc.PROCESSED / "naive_forecasts_2021_2025q3.parquet"
naive_out.to_parquet(naive_path, index=False)
pd.testing.assert_frame_equal(pd.read_parquet(naive_path), naive_out)

# Metrics as CSV: a small table meant to be read by humans and compared later
metrics_path = fc.RESULTS / "06_baseline_metrics.csv"
results.to_csv(metrics_path, index=False)
check = pd.read_csv(metrics_path)
assert np.allclose(check["MAE"], results["MAE"])

print(f"saved {naive_out.shape} -> {naive_path.name}")
print(f"saved {results.shape} -> results/{metrics_path.name}")

## Summary

**Split.** Train Jan 2021 → Sep 2023 (24,071 h), validation Oct 2023 → Sep 2024 (8,784 h), test Oct 2024 → Sep 2025 (8,760 h). The training period contains the gas crisis: in 2021/22 the mean price was 303 EUR/MWh with a standard deviation of 128, against 108 / 30 in validation and 119 / 32 in test. Validation and test belong to the same calm regime, so structure choices made on validation should transfer to the test year.

**Naive forecasts** (MAE in EUR/MWh, validation / test):
- `naive_epf` 12.30 / 12.40 — the benchmark (rMAE = 1 by construction).
- `naive_mean7` 11.89 / 12.60, i.e. rMAE 0.97 / 1.02: better in validation, worse in test. A few percent that flips sign between years is not evidence; a formal Diebold-Mariano test comes with the first model. It does have the lowest RMSE in both years, because averaging smooths out price spikes.
- `naive_d1` 12.58 / 12.59.
- `naive_d7` 14.42 / 14.94 — the worst: one week ago is too far back, level shifts matter more than the weekly cycle.
- Errors are about 11% of the mean price (WAPE 10–12%). Test WAPE is lower than validation at a similar MAE because prices were higher.

**By day of week (validation).** On Mondays `d1` is poor (21.3, it copies Sunday) while `mean7` gets 11.5; Wednesday–Friday the previous day wins (9.4–10.2); on Sundays, Saturday (13.3) beats the previous Sunday (17.4). Calendar features will matter.

**Chosen benchmark:** rMAE against the EPF naive (standard in the literature). A model is only worth keeping if it beats all four naive forecasts.

**Deliberately not done:** combining the best naive per day of week. That would be data snooping; it is a model's job, via calendar features.
